# Section 7.1 Timing Visualizations

This notebook is designed for `exp1` through `exp4` in Section 7.1.

Before running this notebook, generate timing breakdown CSV files with:

- `python experiments/reference_1_section7_1/exp1_live.py --detailed-timing`
- `python experiments/reference_1_section7_1/exp2_live.py --detailed-timing`
- `python experiments/reference_1_section7_1/exp3_live.py --detailed-timing`
- `python experiments/reference_1_section7_1/exp4_live.py --detailed-timing`

Notebook structure:

1. Shared setup and helper cells
2. One dedicated execution cell for `exp1`
3. One dedicated execution cell for `exp2`
4. One dedicated execution cell for `exp3`
5. One dedicated execution cell for `exp4`

For each experiment, this notebook renders 19 plots:

- Three total-runtime plots for `Non-random`, `Random Sampling`, and `Random Projection`
- Two detailed plots for `Non-random`
- Six detailed plots for `Random Sampling`
- Eight detailed plots for `Random Projection`

All plot scales are unified by metric across `exp1` through `exp4`. For example, every `time_sec` plot shares the same y-axis range, every `nr_eig_sec` plot shares the same y-axis range, and so on.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "experiments":
    PROJECT_ROOT = PROJECT_ROOT.parent
EXPERIMENTS_ROOT = PROJECT_ROOT / "experiments"
NOTEBOOK_OUTPUT_ROOT = EXPERIMENTS_ROOT / "timing_visualizations"

METHOD_ORDER = ["Non-random", "Random Sampling", "Random Projection"]
METHOD_COLORS = {
    "Non-random": "#2ca02c",
    "Random Sampling": "#ff7f0e",
    "Random Projection": "#1f77b4",
}

METRIC_LABELS = {
    "time_sec": "Total runtime (sec)",
    "nr_eig_sec": "Top eigenvectors on A (sec)",
    "nr_kmeans_sec": "K-means on eigenvector rows (sec)",
    "rs_sample_mask_sec": "Bernoulli mask generation (sec)",
    "rs_build_sampled_matrix_sec": "Build sampled matrix A_s (sec)",
    "rs_eig_sec": "Top eigenpairs on A_s (sec)",
    "rs_reconstruct_sec": "Rank-K' reconstruction (sec)",
    "rs_symmetrize_sec": "Symmetrization (sec)",
    "rs_kmeans_sec": "K-means on eigenvector rows (sec)",
    "rp_draw_omega_sec": "Draw Gaussian Omega (sec)",
    "rp_power_iter_sec": "Power iteration (sec)",
    "rp_qr_sec": "QR factorization (sec)",
    "rp_build_core_sec": "Build core matrix C (sec)",
    "rp_reconstruct_sec": "Reconstruct A_hat (sec)",
    "rp_small_eig_sec": "Top eigenvectors on C (sec)",
    "rp_lift_sec": "Lift U_rp = Q Uc (sec)",
    "rp_kmeans_sec": "K-means on U_rp rows (sec)",
}

METHOD_STEP_METRICS = {
    "Non-random": [
        "nr_eig_sec",
        "nr_kmeans_sec",
    ],
    "Random Sampling": [
        "rs_sample_mask_sec",
        "rs_build_sampled_matrix_sec",
        "rs_eig_sec",
        "rs_reconstruct_sec",
        "rs_symmetrize_sec",
        "rs_kmeans_sec",
    ],
    "Random Projection": [
        "rp_draw_omega_sec",
        "rp_power_iter_sec",
        "rp_qr_sec",
        "rp_build_core_sec",
        "rp_reconstruct_sec",
        "rp_small_eig_sec",
        "rp_lift_sec",
        "rp_kmeans_sec",
    ],
}

EXPERIMENT_CONFIG = {
    "exp1": {
        "label": "Exp1 (n variation)",
        "summary_filename": "exp1_timing_breakdown_summary.csv",
        "display_cols": ["n"],
        "x_col": "n",
        "x_label": "n",
    },
    "exp2": {
        "label": "Exp2 (alpha_n variation)",
        "summary_filename": "exp2_timing_breakdown_summary.csv",
        "display_cols": ["alpha_n"],
        "x_col": "alpha_n",
        "x_label": "alpha_n",
    },
    "exp3": {
        "label": "Exp3 (K variation)",
        "summary_filename": "exp3_timing_breakdown_summary.csv",
        "display_cols": ["K"],
        "x_col": "K",
        "x_label": "K",
    },
    "exp4": {
        "label": "Exp4 (n variation, alpha_n = 2/sqrt(n))",
        "summary_filename": "exp4_timing_breakdown_summary.csv",
        "display_cols": ["n", "alpha_n"],
        "x_col": "n",
        "x_label": "n",
    },
}

EXPECTED_PLOT_COUNT_PER_EXPERIMENT = 19
ALL_TIMING_METRICS = ["time_sec"]
for method_name in METHOD_ORDER:
    ALL_TIMING_METRICS.extend(METHOD_STEP_METRICS[method_name])


In [ ]:
def find_latest_summary(summary_filename):
    matches = list(EXPERIMENTS_ROOT.glob(f"**/{summary_filename}"))
    if not matches:
        return None
    return max(matches, key=lambda path: path.stat().st_mtime)


def method_slug(method_name):
    return method_name.lower().replace("-", "").replace(" ", "_")


def metric_title(method_name, metric_name):
    return f"{method_name} - {METRIC_LABELS.get(metric_name, metric_name)}"


def build_timing_table(df, base_cols, metrics):
    cols = []
    for col in base_cols:
        if col in df.columns and col not in cols:
            cols.append(col)
    for metric in metrics:
        mean_col = f"{metric}_mean"
        std_col = f"{metric}_std"
        if mean_col in df.columns and mean_col not in cols:
            cols.append(mean_col)
        if std_col in df.columns and std_col not in cols:
            cols.append(std_col)
    table = df.loc[:, cols].copy()
    sort_cols = [col for col in base_cols if col in table.columns]
    if sort_cols:
        table = table.sort_values(sort_cols)
    return table.reset_index(drop=True)


def load_all_summaries():
    loaded = {}
    for exp_key, cfg in EXPERIMENT_CONFIG.items():
        summary_path = find_latest_summary(cfg["summary_filename"])
        if summary_path is None:
            loaded[exp_key] = {"path": None, "df": None}
        else:
            loaded[exp_key] = {"path": summary_path, "df": pd.read_csv(summary_path)}
    return loaded


def compute_global_metric_limits(loaded_summaries):
    limits = {}
    for metric_name in ALL_TIMING_METRICS:
        max_value = 0.0
        mean_col = f"{metric_name}_mean"
        std_col = f"{metric_name}_std"
        for item in loaded_summaries.values():
            df = item["df"]
            if df is None or mean_col not in df.columns:
                continue
            upper = df[mean_col].astype(float)
            if std_col in df.columns:
                upper = upper + df[std_col].astype(float)
            if not upper.empty:
                max_value = max(max_value, float(upper.max()))
        padding = 0.05 * max_value if max_value > 0 else 1.0
        limits[metric_name] = (0.0, max_value + padding)
    return limits


def plot_metric(df, method_name, x_col, x_label, metric_name, exp_label, output_dir, global_limits):
    mean_col = f"{metric_name}_mean"
    std_col = f"{metric_name}_std"
    if mean_col not in df.columns:
        print(f"Skipping {exp_label} / {method_name} / {metric_name}: missing column {mean_col}")
        return None

    method_df = df[df["method"] == method_name].copy()
    if method_df.empty:
        print(f"Skipping {exp_label} / {method_name} / {metric_name}: no rows found")
        return None

    method_df = method_df.sort_values(x_col)
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    color = METHOD_COLORS[method_name]
    ax.plot(
        method_df[x_col],
        method_df[mean_col],
        color=color,
        linewidth=2.2,
        marker="o",
    )
    if std_col in method_df.columns:
        lower = method_df[mean_col] - method_df[std_col]
        upper = method_df[mean_col] + method_df[std_col]
        ax.fill_between(method_df[x_col], lower, upper, color=color, alpha=0.2)
    if metric_name in global_limits:
        ax.set_ylim(*global_limits[metric_name])
    ax.set_title(f"{exp_label} | {metric_title(method_name, metric_name)}")
    ax.set_xlabel(x_label)
    ax.set_ylabel(METRIC_LABELS.get(metric_name, metric_name))
    ax.grid(alpha=0.3)
    fig.tight_layout()

    out_png = output_dir / f"{method_slug(method_name)}_{metric_name}.png"
    fig.savefig(out_png, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return out_png


def render_experiment(exp_key, cfg, loaded_summaries, global_limits):
    display(Markdown(f"## {cfg['label']}"))
    item = loaded_summaries[exp_key]
    summary_path = item["path"]
    df = item["df"]

    if summary_path is None or df is None:
        print(
            "Timing summary CSV not found. "
            f"Expected filename: {cfg['summary_filename']}. "
            "Run the matching experiment with --detailed-timing first."
        )
        return

    print(f"Loaded summary CSV: {summary_path}")
    output_dir = NOTEBOOK_OUTPUT_ROOT / exp_key
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Output directory: {output_dir}")

    display(Markdown("### Timing Tables"))
    overall_table = build_timing_table(df, cfg["display_cols"] + ["method"], ["time_sec"])
    display(Markdown("#### Overall runtime summary"))
    display(overall_table.round(6))

    for method_name in METHOD_ORDER:
        method_df = df[df["method"] == method_name].copy()
        detail_metrics = METHOD_STEP_METRICS[method_name]
        detail_table = build_timing_table(method_df, cfg["display_cols"], detail_metrics)
        display(Markdown(f"#### {method_name} detailed timing summary"))
        display(detail_table.round(6))

    display(Markdown("### Plots"))
    created_paths = []

    for method_name in METHOD_ORDER:
        out_path = plot_metric(
            df=df,
            method_name=method_name,
            x_col=cfg["x_col"],
            x_label=cfg["x_label"],
            metric_name="time_sec",
            exp_label=cfg["label"],
            output_dir=output_dir,
            global_limits=global_limits,
        )
        if out_path is not None:
            created_paths.append(out_path)

    for method_name in METHOD_ORDER:
        for metric_name in METHOD_STEP_METRICS[method_name]:
            out_path = plot_metric(
                df=df,
                method_name=method_name,
                x_col=cfg["x_col"],
                x_label=cfg["x_label"],
                metric_name=metric_name,
                exp_label=cfg["label"],
                output_dir=output_dir,
                global_limits=global_limits,
            )
            if out_path is not None:
                created_paths.append(out_path)

    print(
        f"Prepared {len(created_paths)} plot outputs for {exp_key}. "
        f"Expected: {EXPECTED_PLOT_COUNT_PER_EXPERIMENT}."
    )
    for path in created_paths:
        print(f"  - {path}")


In [ ]:
LOADED_SUMMARIES = load_all_summaries()
GLOBAL_METRIC_LIMITS = compute_global_metric_limits(LOADED_SUMMARIES)

display(Markdown("## Global Y-Axis Limits"))
global_limits_table = pd.DataFrame(
    [
        {
            "metric": metric_name,
            "y_min": y_range[0],
            "y_max": y_range[1],
        }
        for metric_name, y_range in GLOBAL_METRIC_LIMITS.items()
    ]
)
display(global_limits_table.round(6))


In [ ]:
render_experiment("exp1", EXPERIMENT_CONFIG["exp1"], LOADED_SUMMARIES, GLOBAL_METRIC_LIMITS)


In [ ]:
render_experiment("exp2", EXPERIMENT_CONFIG["exp2"], LOADED_SUMMARIES, GLOBAL_METRIC_LIMITS)


In [ ]:
render_experiment("exp3", EXPERIMENT_CONFIG["exp3"], LOADED_SUMMARIES, GLOBAL_METRIC_LIMITS)


In [ ]:
render_experiment("exp4", EXPERIMENT_CONFIG["exp4"], LOADED_SUMMARIES, GLOBAL_METRIC_LIMITS)
